# A侧三项风险证据复核

## tl;dr
21项资源锁历史均不可追溯，当前快照身份21项匹配不等于历史无覆盖。规定名称13项超过历史记录的30字符限制，当前名称均不超过。政策事实2744行，40条2026记录未匹配2010—2025年份维，全部为待人工核验记录；无膨胀不等于无遗漏。原工作簿不改，风险最终接受和B签署均未完成。

## Context & Methods
2026-09-04，Asia/Shanghai。只读现有工作簿和本机快照，不进行平台查询、恢复、CPI或AI测试。先使用同目录inspect_risks.mjs提取原表；verify_risks.py校验源文件哈希并复算。该脚本正文为完整检查逻辑，可供检查。

### Key Assumptions
30字符是8月24日执行日志中的历史说明，本次未重新测试平台限制。年份比较按源表整数键精确匹配；左连接保留未匹配事实，不填0、不删行。结果只证明源数据和已有台账，不自动证明当前页面过滤行为。

## Data
资源锁、A侧/共享两份关系审计、country_policy_year.xlsx及dim_year.xlsx。以下检查绑定源文件哈希；若尚无outputs/READ_ONLY_INPUTS.json，先在具备@oai/artifact-tool的环境执行node inspect_risks.mjs。

验证方式：当前捆绑环境缺nbformat/nbclient，因此本文件以标准库构造nbformat 4结构，全部代码单元用同一Python上下文按顺序执行并保留输出，另作结构断言；没有启动Jupyter内核。

In [1]:
from pathlib import Path
import importlib.util
import json

here = Path.cwd()
assert (here / 'verify_risks.py').exists(), 'Run this notebook from its own folder'
spec = importlib.util.spec_from_file_location('risk_check', here / 'verify_risks.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
report = module.evaluate()
print(json.dumps({k: {'path':v['path'], 'sha256':v['sha256']} for k,v in report['sourceWorkbooks'].items()}, ensure_ascii=False, indent=2))

{
  "lock": {
    "path": "A_数据平台/03_Smartbi证据/A03/SMARTBI_RESOURCE_LOCK_V50.xlsx",
    "sha256": "79f0f5a2bce965f15f09fb6559407d48ebb7141e2454a2151011d218ac9bb205"
  },
  "auditA": {
    "path": "A_数据平台/03_Smartbi证据/模型/RELATIONSHIP_AUDIT_V50.xlsx",
    "sha256": "d448d17f24dcbfd0e0e1dc1171179b4af79644e888b60b23887d4343a5402c26"
  },
  "auditShared": {
    "path": "00_共享/模型交接/RELATIONSHIP_AUDIT_V50.xlsx",
    "sha256": "a997376d94bf20d78647ffc1161b775d6d47b6e2930ea71f9cc48d57e8f2f39d"
  },
  "policy": {
    "path": "A_数据平台/01_输入只读镜像/D0-D12_数据交付_V4.2/data/smartbi/country_policy_year.xlsx",
    "sha256": "b2b773d363967ecec470c57e111e27cd4d835dcb2f9d6dbd4fa752a86cabedfd"
  },
  "years": {
    "path": "A_数据平台/01_输入只读镜像/D0-D12_数据交付_V4.2/data/smartbi/dim_year.xlsx",
    "sha256": "c5e52389428ca3cce4cf10f20627dfeede9db7a413071ff0fbc2d2b88c709ffd"
  }
}


## Results
### 资源历史及命名长度

In [2]:
print(json.dumps(report['resourceHistory'], ensure_ascii=False, indent=2))
print(json.dumps({k:v for k,v in report['naming'].items() if k != 'mapping'}, ensure_ascii=False, indent=2))

{
  "targetCount": 21,
  "historicallyUntraceable": 21,
  "currentSnapshotIdentityMatches": 21,
  "preImportNoOverwriteProven": false,
  "wholeTenantUniquenessProven": false,
  "originalLockWorkbookModified": false
}
{
  "specifiedPrefix": "TB_XH202612_V50_",
  "specifiedPrefixLength": 16,
  "actualPrefix": "V50_",
  "actualPrefixLength": 4,
  "historicallyReportedLengthLimit": 30,
  "platformLimitRetestedToday": false,
  "specifiedOver30": 13,
  "actualOver30": 0,
  "maximumSpecifiedLength": 41,
  "maximumActualLength": 29,
  "archiveGeneratorGuardStaticallyPresent": true,
  "generatorExecuted": false,
  "readOnlyEntryExists": true
}


### 年度政策匹配与原台账

In [3]:
print(json.dumps(report['relation14'], ensure_ascii=False, indent=2))
assert report['relation14']['factRows'] == report['relation14']['sourceInnerJoinRows'] + report['relation14']['unmatchedRows']
assert report['relation14']['sourceLeftJoinRows'] == report['relation14']['factRows']
assert not any(report['boundaries'][k] for k in ('bSigned','finalWaiverGranted','finalFreezeCreated','independentRestoreExecuted','cpiRepairOrRetest','skippedAiRetested'))
print('本地证据复算一致；没有改写源表或新增平台/最终验收PASS。')

{
  "factRows": 2744,
  "factGrain": [
    "iso3",
    "year",
    "policy_code"
  ],
  "dimensionRows": 16,
  "dimensionYearMin": 2010,
  "dimensionYearMax": 2025,
  "sourceInnerJoinRows": 2704,
  "sourceLeftJoinRows": 2744,
  "sourceJoinInflation": 0,
  "unmatchedRows": 40,
  "unmatchedFractionOfAllFactRows": 0.014577259475218658,
  "unmatchedFractionOf2026Rows": 1.0,
  "unmatchedDistinctCountries": 40,
  "unmatchedYears": {
    "2026": 40
  },
  "unmatchedPolicyCodes": {
    "SANCTIONS_OFAC_PROGRAM": 40
  },
  "unmatchedQualityFlags": {
    "review": 40
  },
  "unmatchedReviewStatus": {
    "机提待核(生效日待人工确认)": 40
  },
  "unmatchedValueNullCount": 0,
  "historicalAuditCellRange": "关系审计!A15:H15",
  "historicalAuditRow": [
    14,
    "V50_dim_year → V50_country_policy_year [year_key = year]",
    "V50_country_policy_year",
    2744,
    2704,
    "40(已核验)",
    "无",
    "PASS"
  ],
  "historicalPassDoesNotProveNoOmission": true,
  "liveJoinRetestedToday": false,
  "sourceOrRelationshipM

## Takeaways
- A可确认实现现状与风险披露，不能补签上传前无同名，也不能用本地源表结果替代平台实测。
- 对象别名、视图ID、物理表及源文件联合定位；V50_自身不保证跨项目唯一。旧生成器有默认拒绝重建保护，现行只读入口另行登记。
- 40条差异占全政策表约1.46%，却占2026分区100%。保留原记录与待核标志；不扩只读年份维、不把2026搬到2025、不把匹配行数2704称作全量2744。
- 原关系审计A15:H15的PASS原样保留，本说明明确它不能证明无遗漏或最终豁免。G2既有启动裁决不变，不重列为B执行DB05的前置任务。
- 本次A处置说明见共享区A_RISK_DISPOSITION_20260904.md；最终风险接受、独立恢复和B最终签收未完成。